# Hybrid ALNS Hyperparameter Study: Benchmarking and Tuning Protocol

---

## I. Purpose and Study Design

This notebook extends benchmark execution with a structured hyperparameter search over hybrid-ALNS controls. It is intended for selecting robust configurations, not for one-off anecdotal wins.


## II. Preconditions and Fairness Constraints

Inputs required:
- `repair_model.pkl` from offline training,
- benchmark instances,
- fixed evaluation budget policy.

Fairness constraints:
- each candidate configuration must be tested on the same instance subset,
- seed policy should be consistent (single-seed quick scan, multi-seed confirmation),
- selection must be based on aggregated metrics, not isolated best-case outcomes.


## III. Baseline Reference Run

Establish the non-ML ALNS reference before tuning hybrid parameters. This anchors the magnitude of achievable improvement.


In [ ]:
!python solver.py \
    --instances-dir ../instances \
    --iters 2000 \
    --alns \
    --seed 42 \
    --out baseline_results.csv


## IV. Initial Hybrid Reference Run

Run a default hybrid configuration first to validate integration and establish a pre-tuning reference point.


In [ ]:
!python solver.py \
    --instances-dir ../instances \
    --iters 2000 \
    --alns \
    --ml-repair \
    --model-path repair_model.pkl \
    --seed 42 \
    --out hybrid_results.csv


## V. Baseline-vs-Hybrid Comparison Prior to Tuning

Compute early comparative metrics to verify that hybridization is directionally beneficial before spending compute budget on broader parameter sweeps.


In [ ]:
import pandas as pd

b = pd.read_csv('baseline_results.csv')
h = pd.read_csv('hybrid_results.csv')

key = ['instance'] if 'instance' in b.columns and 'instance' in h.columns else None
if key:
    m = b.merge(h, on=key, suffixes=('_baseline', '_hybrid'))
else:
    m = pd.concat([b.add_suffix('_baseline'), h.add_suffix('_hybrid')], axis=1)

summary = {}
for col in ['bins','objective','runtime_sec']:
    cb, ch = f'{col}_baseline', f'{col}_hybrid'
    if cb in m.columns and ch in m.columns:
        summary[col] = {
            'baseline_mean': m[cb].mean(),
            'hybrid_mean': m[ch].mean(),
            'relative_change_%': 100*(m[ch].mean()-m[cb].mean())/max(abs(m[cb].mean()),1e-9),
        }

pd.DataFrame(summary).T


## VI. Parameter Grid Definition

Define a compact but informative grid covering exploration/exploitation balance, destroy fraction, acceptance temperature, and iteration budget. Prefer logarithmic or coarse-to-fine spacing when sensitivity is unknown.


## 6) Parameter tuning workflow

Use this section to tune ALNS/metaheuristic controls (e.g., destroy fraction, temperature schedule, operator weights) and compare outcomes consistently.


In [ ]:
import itertools, subprocess, shlex

# Example tuning grid (adapt flags to solver.py options available in your version).
grid = {
    'iters': [1000, 2000],
    'seed': [42, 43, 44],
}

runs = []
for iters, seed in itertools.product(grid['iters'], grid['seed']):
    out = f'tune_iters{iters}_seed{seed}.csv'
    cmd = f"python solver.py --instances-dir ../instances --alns --ml-repair --model-path repair_model.pkl --iters {iters} --seed {seed} --out {out}"
    print(cmd)
    subprocess.run(shlex.split(cmd), check=True)
    runs.append(out)

runs


In [ ]:
import pandas as pd

records = []
for f in runs:
    df = pd.read_csv(f)
    rec = {'file': f}
    for c in ['bins','objective','runtime_sec']:
        if c in df.columns:
            rec[f'{c}_mean'] = df[c].mean()
    records.append(rec)

pd.DataFrame(records).sort_values(by=[c for c in ['bins_mean','objective_mean','runtime_sec_mean'] if c in pd.DataFrame(records).columns])


## VIII. Aggregation, Model Selection, and Next-Step Validation

Rank configurations using primary objective quality, then tie-break with runtime stability. Confirm the selected configuration with additional seeds before declaring it final.
